In [14]:
import torch
import torch.nn as nn
from torch.nn import functional as F

import pandas as pd
import numpy as np

In [15]:
dataset = pd.read_csv('https://raw.githubusercontent.com/futurexskill/ml-model-deployment/main/storepurchasedata_large.csv')

In [16]:
dataset.describe()

,Age,Salary,Purchased
count,1554.000000,1554.000000,1554.000000
mean,44.296010,57042.471042,0.694981
std,17.462458,21209.244800,0.460564
min,18.000000,20000.000000,0.000000
25%,27.000000,46000.000000,0.000000
50%,43.000000,60000.000000,1.000000
75%,62.000000,66000.000000,1.000000
max,69.000000,96000.000000,1.000000


In [17]:
dataset.head()

,Age,Salary,Purchased
0,18,20000,0
1,19,22000,0
2,20,24000,0
3,21,28000,0
4,22,60000,1


In [18]:
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:,-1].values

In [19]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size =.20,random_state=0)

In [20]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [21]:
Xtrain_ = torch.from_numpy(X_train).float()
Xtest_ = torch.from_numpy(X_test).float()

In [22]:
Xtrain_

tensor([[-0.0935,  0.2215],
        [ 1.2217, -0.5342],
        [-1.0655,  0.4104],
        ...,
        [ 1.2789,  0.4104],
        [-0.9512,  0.2215],
        [-1.2943, -1.4316]])

In [23]:
ytrain_ = torch.from_numpy(y_train)
ytest_ = torch.from_numpy(y_test)

In [24]:
ytrain_

tensor([1, 1, 1,  ..., 1, 1, 0])

In [25]:
Xtrain_.shape, ytrain_.shape

(torch.Size([1243, 2]), torch.Size([1243]))

In [26]:
Xtest_.shape, ytest_.shape

(torch.Size([311, 2]), torch.Size([311]))

In [27]:
input_size=2
output_size=2
hidden_size=10

In [28]:
class Net(nn.Module):
   def __init__(self):
       super(Net, self).__init__()
       self.fc1 = torch.nn.Linear(input_size, hidden_size)
       self.fc2 = torch.nn.Linear(hidden_size, hidden_size)
       self.fc3 = torch.nn.Linear(hidden_size, output_size)


   def forward(self, X):
       X = torch.relu((self.fc1(X)))
       X = torch.relu((self.fc2(X)))
       X = self.fc3(X)

       return F.log_softmax(X,dim=1)

In [29]:
model = Net()



In [30]:
import torch.optim as optim
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.NLLLoss()

In [31]:
epochs = 100


In [32]:
for epoch in range(epochs):
  optimizer.zero_grad()
  Ypred = model(Xtrain_)
  loss = loss_fn(Ypred,  ytrain_)
  loss.backward()
  optimizer.step()
  print('Epoch',epoch, 'loss',loss.item())

Epoch 0 loss 0.7277731895446777
Epoch 1 loss 0.6965716481208801
Epoch 2 loss 0.666520893573761
Epoch 3 loss 0.6377781629562378
Epoch 4 loss 0.6103225350379944
Epoch 5 loss 0.5842302441596985
Epoch 6 loss 0.5590928196907043
Epoch 7 loss 0.5346207022666931
Epoch 8 loss 0.5108087062835693
Epoch 9 loss 0.48760560154914856
Epoch 10 loss 0.46499326825141907
Epoch 11 loss 0.4432085156440735
Epoch 12 loss 0.4222310483455658
Epoch 13 loss 0.402011901140213
Epoch 14 loss 0.3826345205307007
Epoch 15 loss 0.36415207386016846
Epoch 16 loss 0.34706780314445496
Epoch 17 loss 0.33120930194854736
Epoch 18 loss 0.3159940838813782
Epoch 19 loss 0.3012758493423462
Epoch 20 loss 0.2871514558792114
Epoch 21 loss 0.27365049719810486
Epoch 22 loss 0.2607346475124359
Epoch 23 loss 0.24824044108390808
Epoch 24 loss 0.23634837567806244
Epoch 25 loss 0.225278839468956
Epoch 26 loss 0.2150258719921112
Epoch 27 loss 0.20529448986053467
Epoch 28 loss 0.19642683863639832
Epoch 29 loss 0.1882365196943283
Epoch 30 loss

In [33]:
list(model.parameters())

[Parameter containing:
 tensor([[-0.7372, -0.3654],
         [-0.3513,  0.7026],
         [ 0.4050,  0.9125],
         [ 1.1499, -0.7830],
         [ 0.6371,  0.5725],
         [-1.1786,  0.0422],
         [-0.3560, -0.4895],
         [ 0.6505,  0.2395],
         [-1.0579, -1.0634],
         [-0.8702, -0.8059]], requires_grad=True),
 Parameter containing:
 tensor([ 0.2103,  0.5438,  0.4283, -0.2770,  0.7895, -0.1289,  0.2012,  0.8536,
          0.0211,  0.0307], requires_grad=True),
 Parameter containing:
 tensor([[ 0.2777,  0.6541,  0.4024,  0.7797,  0.5112,  0.4978, -0.2478,  0.4764,
          -0.2130, -0.3932],
         [ 0.1509, -0.0792, -0.0455, -0.3471, -0.1022, -0.6219,  0.0231, -0.0710,
           0.2373,  0.5484],
         [ 0.4037,  0.3230,  0.1394,  0.5526,  0.1367,  0.4813,  0.0128,  0.4661,
          -0.5261, -0.4811],
         [ 0.6603,  0.3091, -0.7056, -0.1154, -0.2198, -0.5940,  0.5801,  0.2226,
           0.6044,  0.6118],
         [-0.2953,  0.7165,  0.6453,  1.0459,

In [35]:
torch.from_numpy(sc.transform(np.array([[40,20000]]))).float()

tensor([[-0.2650, -1.7622]])

In [38]:
y_cust_20_40000 = model(torch.from_numpy(sc.transform(np.array([[42,50000]]))).float())
y_cust_20_40000

tensor([[-0.7553, -0.6346]], grad_fn=<LogSoftmaxBackward0>)

In [39]:
_, predicted_20_40000 = torch.max(y_cust_20_40000.data,-1)
predicted_20_40000

tensor([1])

In [40]:
y_cust_42_50000 = model(torch.from_numpy(sc.transform(np.array([[42,50000]]))).float())
y_cust_42_50000

tensor([[-0.7553, -0.6346]], grad_fn=<LogSoftmaxBackward0>)

In [41]:
_, predicted_42_50000 = torch.max(y_cust_42_50000.data,-1)
predicted_42_50000

tensor([1])

In [53]:
torch.save(model,'customer_buy.pt')

In [51]:
!ls

'ls' is not recognized as an internal or external command,
operable program or batch file.


In [54]:
restored_model = torch.load('customer_buy.pt',weights_only=False)

In [55]:
y_cust_20_40000 = restored_model(torch.from_numpy(sc.transform(np.array([[40,20000]]))).float())
y_cust_20_40000

tensor([[-4.9269e-03, -5.3155e+00]], grad_fn=<LogSoftmaxBackward0>)

In [56]:
_, predicted_20_40000 = torch.max(y_cust_20_40000.data,-1)
predicted_20_40000

tensor([0])

In [57]:
model.state_dict()

OrderedDict([('fc1.weight',
              tensor([[-0.7372, -0.3654],
                      [-0.3513,  0.7026],
                      [ 0.4050,  0.9125],
                      [ 1.1499, -0.7830],
                      [ 0.6371,  0.5725],
                      [-1.1786,  0.0422],
                      [-0.3560, -0.4895],
                      [ 0.6505,  0.2395],
                      [-1.0579, -1.0634],
                      [-0.8702, -0.8059]])),
             ('fc1.bias',
              tensor([ 0.2103,  0.5438,  0.4283, -0.2770,  0.7895, -0.1289,  0.2012,  0.8536,
                       0.0211,  0.0307])),
             ('fc2.weight',
              tensor([[ 0.2777,  0.6541,  0.4024,  0.7797,  0.5112,  0.4978, -0.2478,  0.4764,
                       -0.2130, -0.3932],
                      [ 0.1509, -0.0792, -0.0455, -0.3471, -0.1022, -0.6219,  0.0231, -0.0710,
                        0.2373,  0.5484],
                      [ 0.4037,  0.3230,  0.1394,  0.5526,  0.1367,  0.4813,  0.0128

In [58]:
torch.save(model.state_dict(),'customer_buy_state_dict')

In [59]:
!ls

'ls' is not recognized as an internal or external command,
operable program or batch file.


In [60]:
new_predictor = Net()

In [61]:
y_cust_20_40000 = new_predictor(torch.from_numpy(sc.transform(np.array([[40,20000]]))).float())
y_cust_20_40000

tensor([[-0.5109, -0.9162]], grad_fn=<LogSoftmaxBackward0>)

'zip' is not recognized as an internal or external command,
operable program or batch file.


In [37]:
!ls

customer_buy.pt		 customer_buy_state_dict.zip
customer_buy_state_dict  sample_data


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>